# DPCexplorer API example: fetching metaclusters by MCID

This notebook shows how to pull DPCfam or DPCstruct metacluster data from
the DPCexplorer REST API into a pandas DataFrame, using either one MCID or
a list of MCIDs.

No login is required: the API is public and read-only. See
`DPCexplorer_API_Documentation.md` in the repository for the full
endpoint reference.

In [1]:
# Installations
%pip install requests pandas
# Clean installation outputs
from IPython.display import clear_output
clear_output()

In [ ]:
import requests
import pandas as pd

# URL of the API
# (A) Production
BASE_URL = "https://dpcexplorer.areasciencepark.it/api"
# (B) Local development server
# BASE_URL = "http://127.0.0.1:8000/api"

## 1. Choose a dataset and one or more MCIDs

Edit the two variables below. `DATASET` must be `"dpcfam"` or
`"dpcstruct"`. `MCIDS` accepts a single MCID or a list.

In [3]:
DATASET = "dpcstruct"              # "dpcfam" or "dpcstruct"
MCIDS = ["MC0", "MC1", "MC64574"]  # a single MCID also works: MCIDS = ["MC1"]

## 2. Fetch properties for all requested MCIDs in one call

In [4]:
def fetch_properties(dataset, mcids):
    url = f"{BASE_URL}/{dataset}/mcs/"
    params = {"mcids": ",".join(mcids)}
    properties = []
    while url:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        properties.extend(payload["results"])
        url = payload["next"]
        params = None
    return properties

properties = fetch_properties(DATASET, MCIDS)
properties_df = pd.DataFrame(properties)
properties_df

,mc_id,mc_size,len_aa,len_std,len_ratio,plddt,disorder,tmscore,lddt,pident,pfam_score,pfam_da
0,MC0,43,112.02,12.36,0.11,78.53,0.29,0.54,0.65,23.74,0.0,UNKNOWN
1,MC1,19,68.89,7.43,0.11,90.81,0.28,0.77,0.82,25.87,0.0,UNKNOWN
2,MC64574,11,90.36,9.42,0.10,86.57,0.20,0.87,0.84,39.65,0.0,UNKNOWN


## 3. Fetch members (ID, Protein ID, Range) for one metacluster

Members are paginated. This cell requests up to 500 members per page,
follows every page for one MCID, and concatenates the results.

In [5]:
def fetch_all_members(dataset, mcid):
    url = f"{BASE_URL}/{dataset}/mcs/{mcid}/members/"
    params = {"page_size": 500}
    all_members = []
    while url:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        all_members.extend(payload["results"])
        url = payload["next"]
        params = None
    return all_members

mcid_to_inspect = MCIDS[0]
members = fetch_all_members(DATASET, mcid_to_inspect)
members_df = pd.DataFrame(members)
print(f"{mcid_to_inspect}: {len(members_df)} member(s)")
members_df.head()

MC0: 43 member(s)


,id,protein_id,prot_range
0,1,A0A1Q3ZAL7,4-118
1,2,A0A537IMV5,3-112
2,3,A0A170YWQ7,2-107
3,4,A0A3E2NVG7,1-133
4,5,A0A2W4V3S2,3-113


## 4. Save results locally

In [6]:
properties_df.to_csv(f"{DATASET}_properties.csv", index=False)
members_df.to_csv(f"{DATASET}_{mcid_to_inspect}_members.csv", index=False)
print("Saved CSV files in the current directory.")

Saved CSV files in the current directory.
